In [ ]:
pip install torch torchvision opencv-python numpy matplotlib tqdm

In [ ]:
import os
import gc
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

# ==========================================
# 0. CONFIGURATION & HYPERPARAMETERS
# ==========================================
class Config:
    # Set to True to create a synthetic high-res image for testing in Kaggle
    USE_SYNTHETIC_IMAGE = True
    SYNTHETIC_SIZE = (12000, 12000)  # Simulated slide dimensions (H, W)
    
    # Patching Parameters
    PATCH_SIZE = 256        # Size of extracted patches (256x256)
    STRIDE = 256            # Stride for non-overlapping tiling (set lower for overlap)
    THUMB_SCALE = 0.05      # Scale factor for thumbnail processing (5% resolution)
    TISSUE_THRESHOLD = 0.15 # Min proportion of tissue pixels required per patch
    
    # DL Parameters
    BATCH_SIZE = 64
    NUM_WORKERS = 2
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Visualization
    HEATMAP_ALPHA = 0.45


# ==========================================
# 1. TISSUE FILTERING & PREPROCESSING
# ==========================================
class TissueSegmenter:
    """Extracts tissue foreground mask from ultra-high-resolution slide thumbnail."""
    
    @staticmethod
    def generate_tissue_mask(thumb_rgb: np.ndarray) -> np.ndarray:
        """
        Applies HSV saturation thresholding and Otsu thresholding to segment tissue from background.
        """
        # Convert thumbnail to HSV color space
        hsv = cv2.cvtColor(thumb_rgb, cv2.COLOR_RGB2HSV)
        s_channel = hsv[:, :, 1]  # Saturation channel separates tissue dye from white background
        
        # Otsu's thresholding on saturation
        _, mask = cv2.threshold(s_channel, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        # Morphological opening/closing to clean noise and fill small holes
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        
        return mask


# ==========================================
# 2. SCALABLE PATCH EXTRACTION
# ==========================================
class PatchExtractor:
    """Generates coordinate grids and filters out empty background patches using tissue mask."""
    
    def __init__(self, img_shape, patch_size=256, stride=256, thumb_scale=0.05):
        self.img_h, self.img_w = img_shape[:2]
        self.patch_size = patch_size
        self.stride = stride
        self.scale = thumb_scale

    def get_valid_patch_coords(self, tissue_mask: np.ndarray, tissue_thresh: float):
        valid_coords = []
        mask_h, mask_w = tissue_mask.shape
        
        # Iterate across high-res image coordinate space
        for y in range(0, self.img_h - self.patch_size + 1, self.stride):
            for x in range(0, self.img_w - self.patch_size + 1, self.stride):
                
                # Map high-res coordinates to downsampled mask space
                my1 = int(y * self.scale)
                my2 = int((y + self.patch_size) * self.scale)
                mx1 = int(x * self.scale)
                mx2 = int((x + self.patch_size) * self.scale)
                
                # Ensure boundary safety
                my2 = min(my2, mask_h)
                mx2 = min(mx2, mask_w)
                
                patch_mask = tissue_mask[my1:my2, mx1:mx2]
                if patch_mask.size == 0:
                    continue
                    
                tissue_ratio = np.mean(patch_mask > 0)
                
                # Retain tile if it contains sufficient tissue ratio
                if tissue_ratio >= tissue_thresh:
                    # Store (y, x) top-left image coords and grid matrix index (grid_y, grid_x)
                    grid_y = y // self.stride
                    grid_x = x // self.stride
                    valid_coords.append({
                        'x': x, 'y': y,
                        'grid_x': grid_x, 'grid_y': grid_y
                    })
                    
        return valid_coords


# ==========================================
# 3. LAZY-LOADING PYTORCH DATASET
# ==========================================
class PathologyPatchDataset(Dataset):
    """
    Lazy-loads individual patches from memory/disk to prevent RAM spikes.
    """
    def __init__(self, image_source, coords, patch_size=256, transform=None):
        self.image_source = image_source
        self.coords = coords
        self.patch_size = patch_size
        self.transform = transform

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        c = self.coords[idx]
        x, y = c['x'], c['y']
        
        # Crop patch on demand
        patch = self.image_source[y:y+self.patch_size, x:x+self.patch_size]
        
        # Convert to PIL for Torchvision Transforms
        patch_pil = Image.fromarray(patch)
        
        if self.transform:
            patch_tensor = self.transform(patch_pil)
        else:
            patch_tensor = T.ToTensor()(patch_pil)
            
        return patch_tensor, c['grid_y'], c['grid_x']


# ==========================================
# 4. DEEP LEARNING MODEL & INFERENCE ENGINE
# ==========================================
def build_pathology_classifier():
    """Initializes a ResNet18 backbone modified for binary anomaly classification."""
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_ftrs, 128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, 1),
        nn.Sigmoid()
    )
    return model


def run_batch_inference(model, dataloader, device):
    """Executes high-throughput, memory-efficient inference using Mixed Precision."""
    model.to(device)
    model.eval()
    
    results = []
    
    with torch.no_grad():
        for patches, grid_ys, grid_xs in tqdm(dataloader, desc="Running Deep Learning Inference"):
            patches = patches.to(device, non_blocking=True)
            
            # Use Automatic Mixed Precision for fast computation
            with torch.amp.autocast(device_type="cuda" if "cuda" in device else "cpu"):
                outputs = model(patches).squeeze(-1)
                
            probs = outputs.cpu().numpy()
            grid_ys = grid_ys.numpy()
            grid_xs = grid_xs.numpy()
            
            for p, gy, gx in zip(probs, grid_ys, grid_xs):
                results.append((gy, gx, float(p)))
                
    return results


# ==========================================
# 5. HEATMAP RECONSTRUCTION & LOCALIZATION
# ==========================================
class HeatmapReconstructor:
    """Reconstructs patch predictions into a low-resolution WSI probability map."""
    
    @staticmethod
    def build_probability_map(results, grid_h, grid_w):
        prob_map = np.zeros((grid_h, grid_w), dtype=np.float32)
        for gy, gx, prob in results:
            prob_map[gy, gx] = prob
        return prob_map

    @staticmethod
    def overlay_heatmap(thumb_rgb, prob_map, threshold=0.5, alpha=0.45):
        # Resize probability map to match thumbnail dimensions
        h, w, _ = thumb_rgb.shape
        heatmap_resized = cv2.resize(prob_map, (w, h), interpolation=cv2.INTER_CUBIC)
        
        # Smooth and convert to 8-bit heatmap image
        heatmap_resized = np.clip(heatmap_resized, 0, 1)
        heatmap_uint8 = (heatmap_resized * 255).astype(np.uint8)
        color_heatmap = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
        color_heatmap = cv2.cvtColor(color_heatmap, cv2.COLOR_BGR2RGB)
        
        # Blend with thumbnail
        overlay = cv2.addWeighted(thumb_rgb, 1 - alpha, color_heatmap, alpha, 0)
        
        # Localize abnormal regions via contour detection on thresholded heatmap
        binary_abnormal = (heatmap_resized >= threshold).astype(np.uint8) * 255
        contours, _ = cv2.findContours(binary_abnormal, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        annotated_overlay = overlay.copy()
        bbox_list = []
        for cnt in contours:
            if cv2.contourArea(cnt) > 20: # Filter small noise
                bx, by, bw, bh = cv2.boundingRect(cnt)
                cv2.rectangle(annotated_overlay, (bx, by), (bx + bw, by + bh), (255, 0, 0), 2)
                bbox_list.append((bx, by, bw, bh))
                
        return heatmap_resized, overlay, annotated_overlay, bbox_list


# ==========================================
# MOCK DATA GENERATOR (For Kaggle Testing)
# ==========================================
def create_synthetic_wsi(height=12000, width=12000):
    """Generates a synthetic ultra-high-resolution canvas simulating tissue and lesions."""
    print(f"Creating synthetic WSI of shape ({height}, {width}, 3)...")
    # Background: White Glass (~240 RGB)
    canvas = np.full((height, width, 3), 242, dtype=np.uint8)
    
    # Draw Pink/Purple Tissue Regions
    cv2.ellipse(canvas, (width//3, height//3), (3000, 2000), 30, 0, 360, (180, 80, 160), -1)
    cv2.ellipse(canvas, (2*width//3, 2*height//3), (2500, 2500), 0, 0, 360, (160, 60, 140), -1)
    
    # Add noise / texture
    noise = np.random.randint(-15, 15, canvas.shape, dtype=np.int16)
    canvas = np.clip(canvas.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return canvas


# ==========================================
# 6. MAIN EXECUTION PIPELINE
# ==========================================
def main():
    print(f"Executing Pathology Analysis Pipeline on Device: {Config.DEVICE}")
    
    # 1. Load Image / Generate Synthetic Ultra-High-Res Image
    if Config.USE_SYNTHETIC_IMAGE:
        wsi_image = create_synthetic_wsi(*Config.SYNTHETIC_SIZE)
    else:
        # Load real high-res image (e.g. via cv2 or memory-mapped array)
        wsi_image = cv2.imread("path_to_slide.tif")
        wsi_image = cv2.cvtColor(wsi_image, cv2.COLOR_BGR2RGB)
        
    img_h, img_w, _ = wsi_image.shape
    print(f"Slide Dimensions: {img_w}x{img_h} pixels (~{(img_w*img_h)/1e6:.1f} MP)")

    # 2. Downsample for Thumbnail Processing
    thumb_w = int(img_w * Config.THUMB_SCALE)
    thumb_h = int(img_h * Config.THUMB_SCALE)
    thumb_rgb = cv2.resize(wsi_image, (thumb_w, thumb_h), interpolation=cv2.INTER_AREA)

    # 3. Tissue Mask Generation
    print("Generating tissue segmentation mask...")
    tissue_mask = TissueSegmenter.generate_tissue_mask(thumb_rgb)

    # 4. Patch Coordinate Extraction & Background Filtering
    print("Extracting valid patch coordinates...")
    extractor = PatchExtractor(
        img_shape=(img_h, img_w),
        patch_size=Config.PATCH_SIZE,
        stride=Config.STRIDE,
        thumb_scale=Config.THUMB_SCALE
    )
    valid_coords = extractor.get_valid_patch_coords(tissue_mask, Config.TISSUE_THRESHOLD)
    
    grid_h = (img_h - Config.PATCH_SIZE) // Config.STRIDE + 1
    grid_w = (img_w - Config.PATCH_SIZE) // Config.STRIDE + 1
    total_possible_tiles = grid_h * grid_w
    
    print(f"Grid Matrix Size: {grid_h} x {grid_w}")
    print(f"Filtered Patches: Retained {len(valid_coords)} / {total_possible_tiles} tiles "
          f"({(len(valid_coords)/total_possible_tiles)*100:.1f}% active tissue).")

    # 5. Data Transformations & PyTorch DataLoader
    transform = T.Compose([
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    dataset = PathologyPatchDataset(
        image_source=wsi_image,
        coords=valid_coords,
        patch_size=Config.PATCH_SIZE,
        transform=transform
    )
    
    dataloader = DataLoader(
        dataset,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        num_workers=Config.NUM_WORKERS,
        pin_memory=True if Config.DEVICE == "cuda" else False
    )

    # 6. Deep Learning Inference
    model = build_pathology_classifier()
    results = run_batch_inference(model, dataloader, Config.DEVICE)

    # 7. Heatmap Reconstruction & Overlay Analysis
    print("Reconstructing whole-slide classification heatmaps...")
    prob_map = HeatmapReconstructor.build_probability_map(results, grid_h, grid_w)
    heatmap_resized, overlay, annotated, bboxes = HeatmapReconstructor.overlay_heatmap(
        thumb_rgb, prob_map, threshold=0.5, alpha=Config.HEATMAP_ALPHA
    )
    print(f"Detected {len(bboxes)} abnormal candidate ROI bounding boxes.")

    # 8. Visualization Output in Kaggle
    fig, axes = plt.subplots(1, 4, figsize=(22, 6))
    
    axes[0].imshow(thumb_rgb)
    axes[0].set_title("1. Original Slide Thumbnail")
    axes[0].axis("off")
    
    axes[1].imshow(tissue_mask, cmap="gray")
    axes[1].set_title("2. Tissue Foreground Mask")
    axes[1].axis("off")
    
    axes[2].imshow(heatmap_resized, cmap="jet")
    axes[2].set_title("3. Probability Heatmap")
    axes[2].axis("off")
    
    axes[3].imshow(annotated)
    axes[3].set_title("4. ROI Detection Overlay")
    axes[3].axis("off")
    
    plt.tight_layout()
    plt.show()

    # Free memory
    del wsi_image
    gc.collect()

if __name__ == "__main__":
    main()